<a href="https://colab.research.google.com/github/AdityaLaddha47/NCPOR-BITS/blob/Wrf/WRF_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# WRF Pipeline — Temperature Forecast
**Run cells in order. On first use, Cell 3 (compile) takes ~60 min. After that, skip to Cell 4.**

## Cell 1: Mount Google Drive
Always run this first, every session.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/WRF', exist_ok=True)
print('Drive mounted. WRF folder ready at /content/drive/MyDrive/WRF')

Mounted at /content/drive
Drive mounted. WRF folder ready at /content/drive/MyDrive/WRF


## Cell 2: Install Dependencies
Run every session (~3 min). Colab resets installed packages on reconnect.

In [2]:
%%bash
apt-get update -q
apt-get install -y -q gcc gfortran g++ make m4 \
  libhdf5-dev libnetcdf-dev libnetcdff-dev netcdf-bin \
  libpng-dev zlib1g-dev perl csh mpich
pip install -q xarray netCDF4 wrf-python matplotlib numpy
echo 'Dependencies installed successfully'

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:10 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,297 kB]
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,090 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,959 kB]
Get:14 h

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


## Cell 3: Compile WRF (FIRST TIME ONLY ~60 min)
**Skip this cell if `/content/drive/MyDrive/WRF/WRFV4.6.0/main/wrf.exe` already exists.**

Keep this browser tab active while compiling — move your mouse occasionally to prevent Colab from disconnecting.

In [3]:
import os

wrf_exe = '/content/drive/MyDrive/WRF/WRFV4.6.0/main/wrf.exe'

if os.path.exists(wrf_exe) and os.path.getsize(wrf_exe) > 1000000:
    print('WRF already compiled on Drive. Skip this cell.')
else:
    print('Starting WRF compilation... (~60 min, keep tab active)')

Starting WRF compilation... (~60 min, keep tab active)


In [5]:
%%bash
# Only run if Cell 3 check above said 'Starting WRF compilation'
cd /content

# Download WRF
wget -q https://github.com/wrf-model/WRF/releases/download/v4.6.0/v4.6.0.tar.gz
tar -xzf v4.6.0.tar.gz
cd WRFV4.6.0

# Set compilers
export FC=gfortran
export CC=gcc
export CXX=g++
export F77=gfortran
export NETCDF=/usr
export NETCDF_classic=1

# Configure (34 = GNU dmpar, 1 = basic nesting)
echo -e '34\n1' | ./configure

# Compile with 2 parallel jobs
./compile -j 2 em_real >& compile.log

# Check result
if [ -f main/wrf.exe ] && [ -f main/real.exe ]; then
    echo 'COMPILATION SUCCESS'
    # Save to Drive
    echo 'Saving to Google Drive (~5-10 min)...'
    cp -r /content/WRFV4.6.0 /content/drive/MyDrive/WRF/
    echo 'Saved to Drive successfully'
else
    echo 'COMPILATION FAILED - check compile.log'
    tail -30 compile.log
fi

checking for perl5... no
checking for perl... found /usr/bin/perl (perl)
Will use NETCDF in dir: /usr
ADIOS2 not set in environment. Will configure WRF for use without.
HDF5 not set in environment. Will configure WRF for use without.
PHDF5 not set in environment. Will configure WRF for use without.
$JASPERLIB or $JASPERINC not found in environment, configuring to build without grib2 I/O...
------------------------------------------------------------------------
Please select from among the following Linux x86_64 options:

  1. (serial)   2. (smpar)   3. (dmpar)   4. (dm+sm)   PGI (pgf90/gcc)
  5. (serial)   6. (smpar)   7. (dmpar)   8. (dm+sm)   PGI (pgf90/pgcc): SGI MPT
  9. (serial)  10. (smpar)  11. (dmpar)  12. (dm+sm)   PGI (pgf90/gcc): PGI accelerator
 13. (serial)  14. (smpar)  15. (dmpar)  16. (dm+sm)   INTEL (ifort/icc)
                                         17. (dm+sm)   INTEL (ifort/icc): Xeon Phi (MIC architecture)
 18. (serial)  19. (smpar)  20. (dmpar)  21. (dm+sm)   IN


gzip: stdin: unexpected end of file
tar: Unexpected EOF in archive
tar: Unexpected EOF in archive
tar: Error is not recoverable: exiting now
./configure: 903: cannot create tools/fortran_2003_ieee_test.log: Directory nonexistent
./configure: 923: cannot create tools/fortran_2003_iso_c_test.log: Directory nonexistent
./configure: 944: cannot create tools/fortran_2003_flush_test.log: Directory nonexistent
./configure: 951: cannot create tools/fortran_2003_fflush_test.log: Directory nonexistent
./configure: 986: cannot create tools/fortran_2008_gamma.log: Directory nonexistent
./configure: 1007: cannot create tools/rpc_test.log: Directory nonexistent


In [7]:
# updates cell 3
%%bash
cd /content

# Remove any incomplete download
rm -f v4.6.0.tar.gz
rm -rf WRFV4.6.0

# Download with retry
wget --tries=3 --timeout=60 \
  https://github.com/wrf-model/WRF/releases/download/v4.6.0/v4.6.0.tar.gz

# Verify the file is complete before extracting
if gzip -t v4.6.0.tar.gz 2>/dev/null; then
    echo "Download OK, extracting..."
    tar -xzf v4.6.0.tar.gz
else
    echo "DOWNLOAD CORRUPT - re-run this cell"
    exit 1
fi

cd WRFV4.6.0
export FC=gfortran CC=gcc CXX=g++ F77=gfortran
export NETCDF=/usr NETCDF_classic=1

echo -e '34\n1' | ./configure
./compile -j 2 em_real >& compile.log

if [ -f main/wrf.exe ] && [ -f main/real.exe ]; then
    echo 'COMPILATION SUCCESS'
    cp -r /content/WRFV4.6.0 /content/drive/MyDrive/WRF/
    echo 'Saved to Drive'
else
    echo 'COMPILATION FAILED'
    tail -20 compile.log
fi

Download OK, extracting...
checking for perl5... no
checking for perl... found /usr/bin/perl (perl)
Will use NETCDF in dir: /usr
ADIOS2 not set in environment. Will configure WRF for use without.
HDF5 not set in environment. Will configure WRF for use without.
PHDF5 not set in environment. Will configure WRF for use without.
$JASPERLIB or $JASPERINC not found in environment, configuring to build without grib2 I/O...
------------------------------------------------------------------------
Please select from among the following Linux x86_64 options:

  1. (serial)   2. (smpar)   3. (dmpar)   4. (dm+sm)   PGI (pgf90/gcc)
  5. (serial)   6. (smpar)   7. (dmpar)   8. (dm+sm)   PGI (pgf90/pgcc): SGI MPT
  9. (serial)  10. (smpar)  11. (dmpar)  12. (dm+sm)   PGI (pgf90/gcc): PGI accelerator
 13. (serial)  14. (smpar)  15. (dmpar)  16. (dm+sm)   INTEL (ifort/icc)
                                         17. (dm+sm)   INTEL (ifort/icc): Xeon Phi (MIC architecture)
 18. (serial)  19. (smpar)  20

--2026-05-30 09:09:44--  https://github.com/wrf-model/WRF/releases/download/v4.6.0/v4.6.0.tar.gz
Resolving github.com (github.com)... 140.82.116.4
Connecting to github.com (github.com)|140.82.116.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/65852069/7d4abcd0-66c9-4589-9bd2-14c00a2f9e2e?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-05-30T09%3A44%3A07Z&rscd=attachment%3B+filename%3Dv4.6.0.tar.gz&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-05-30T08%3A43%3A46Z&ske=2026-05-30T09%3A44%3A07Z&sks=b&skv=2018-11-09&sig=4%2BsH6AVy%2B%2BBdq6YTy24jvak%2BahwZwC0C95TKKDtoqu0%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc4MDEzMzk4NCwibmJmIjoxNzgwMTMyMTg0LCJwYXRoIjoicmVsZWFzZWFzc2V0cHJvZHVjdGlvbi5ibG9iL

In [8]:
import os
wrf = '/content/WRFV4.6.0/main/wrf.exe'
real = '/content/WRFV4.6.0/main/real.exe'

for f in [wrf, real]:
    if os.path.exists(f):
        print(f'{f}: {os.path.getsize(f)/1e6:.1f} MB')
    else:
        print(f'{f}: NOT FOUND')

/content/WRFV4.6.0/main/wrf.exe: NOT FOUND
/content/WRFV4.6.0/main/real.exe: NOT FOUND


In [9]:
import subprocess
result = subprocess.run(['tail', '-30', '/content/WRFV4.6.0/compile.log'],
                      capture_output=True, text=True)
print(result.stdout)


             ln -sf ../../run/SBM_input_33 . ;						\
             ln -sf ../../run/scattering_tables_2layer_high_quad_1dT_1%fw_110 . ;	\
             fi )
( cd test/em_real ; /bin/rm -f GENPARM.TBL ; ln -s ../../run/GENPARM.TBL . )
( cd test/em_real ; /bin/rm -f LANDUSE.TBL ; ln -s ../../run/LANDUSE.TBL . )
( cd test/em_real ; /bin/rm -f SOILPARM.TBL ; ln -s ../../run/SOILPARM.TBL . )
( cd test/em_real ; /bin/rm -f URBPARM.TBL ; ln -s ../../run/URBPARM.TBL . )
( cd test/em_real ; /bin/rm -f URBPARM_LCZ.TBL ; ln -s ../../run/URBPARM_LCZ.TBL . )
( cd test/em_real ; /bin/rm -f VEGPARM.TBL ; ln -s ../../run/VEGPARM.TBL . )
( cd test/em_real ; /bin/rm -f MPTABLE.TBL ; ln -s ../../run/MPTABLE.TBL . )
( cd test/em_real ; /bin/rm -f tr49t67 ; ln -s ../../run/tr49t67 . )
( cd test/em_real ; /bin/rm -f tr49t85 ; ln -s ../../run/tr49t85 . )
( cd test/em_real ; /bin/rm -f tr67t85 ; ln -s ../../run/tr67t85 . )
( cd test/em_real ; /bin/rm -f gribmap.txt ; ln -s ../../run/gribmap.txt . )
( cd test/e

In [10]:
import subprocess
result = subprocess.run(['free', '-h'], capture_output=True, text=True)
print(result.stdout)


               total        used        free      shared  buff/cache   available
Mem:            12Gi       724Mi       9.3Gi       3.0Mi       2.7Gi        11Gi
Swap:             0B          0B          0B



In [11]:
result = subprocess.run(['grep', '-i', 'error\|killed\|terminated',
                        '/content/WRFV4.6.0/compile.log'],
                       capture_output=True, text=True)
print(result.stdout[-3000:])


tng_iccg.o module_fdda_psufddagd.o module_fdda_spnudging.o module_fddagd_driver.o module_fddaobs_rtfdda.o module_fddaobs_driver.o module_wind_fitch.o module_wind_mav.o module_sf_lake.o module_diagnostics_driver.o module_irrigation.o   \
rm -f module_cam_error_function.o
sed -e "s/^\!.*'.*//" -e "s/^ *\!.*'.*//" module_cam_error_function.F > module_cam_error_function.G
/lib/cpp -P -nostdinc -I/content/WRFV4.6.0/inc -DEM_CORE=1 -DNMM_CORE=0 -DNMM_MAX_DIM=2600 -DDA_CORE=0 -DWRFPLUS=0 -DIWORDSIZE=4 -DDWORDSIZE=8 -DRWORDSIZE=4 -DLWORDSIZE=4 -DNONSTANDARD_SYSTEM_SUBR  -DWRF_USE_CLM  -DDM_PARALLEL -DNETCDF -DLANDREAD_STUB=1 -DUSE_ALLOCATABLES -Dwrfmodel -DGRIB1 -DINTIO -DKEEP_INT_AROUND -DLIMIT_ARGS -DBUILD_RRTMG_FAST=0 -DBUILD_RRTMK=0 -DBUILD_SBM_FAST=1 -DSHOW_ALL_VARS_USED=0 -DCONFIG_BUF_LEN=65536 -DMAX_DOMAINS_F=21 -DMAX_HISTORY=25 -DNMM_NEST=0  -I. -traditional-cpp   module_cam_error_function.G  > module_cam_error_function.bb
/content/WRFV4.6.0/tools/standard.exe module_cam_error_function

<>:1: SyntaxWarning: invalid escape sequence '\|'
<>:1: SyntaxWarning: invalid escape sequence '\|'
/tmp/ipykernel_635/3221826165.py:1: SyntaxWarning: invalid escape sequence '\|'
  result = subprocess.run(['grep', '-i', 'error\|killed\|terminated',


In [12]:
%%bash
cd /content/WRFV4.6.0

# Check what netcdf libraries are available
nc-config --libs
nf-config --libs

# Add the Fortran NetCDF library explicitly
export NETCDF=/usr
export NETCDF_classic=1
export FC=gfortran
export CC=gcc
export CXX=g++
export NETCDF_LIB=$(nf-config --flibs)

# Edit configure.wrf to add -lnetcdff to linker flags
sed -i 's|-lwrfio_nf -L/usr/lib|-lwrfio_nf -L/usr/lib -lnetcdff -lnetcdf|g' configure.wrf

# Recompile (only the linking step needs to redo, much faster ~5 min)
./compile -j 2 em_real >& compile.log

# Check result
ls -lh main/wrf.exe main/real.exe 2>/dev/null || (echo "FAILED" && tail -10 compile.log)


-L/usr/lib/x86_64-linux-gnu -L/usr/lib/x86_64-linux-gnu/hdf5/serial -lnetcdf
unknown option: --libs
Usage: nf-config [OPTION]

Available values for OPTION include:

  --help        display this help message and exit
  --all         display all options
  --cc          C compiler
  --fc          Fortran compiler
  --cflags      pre-processor and compiler flags
  --fflags      flags needed to compile a Fortran program
  --has-dap     whether OPeNDAP is enabled in this build
  --has-nc2     whether NetCDF-2 API is enabled
  --has-nc4     whether NetCDF-4/HDF-5 is enabled in this build
  --has-f90     whether Fortran 90 API is enabled in this build
  --has-f03     whether Fortran 2003 API is enabled in this build
  --flibs       libraries needed to link a Fortran program
  --prefix      Install prefix
  --includedir  Include directory
  --version     Library version

-rwxr-xr-x 1 root root 47M May 30 10:10 main/real.exe
-rwxr-xr-x 1 root root 55M May 30 10:10 main/wrf.exe


In [13]:
%%bash
echo 'Saving to Google Drive (~5-10 min)...'
cp -r /content/WRFV4.6.0 /content/drive/MyDrive/WRF/
echo 'Done!'
ls -lh /content/drive/MyDrive/WRF/WRFV4.6.0/main/*.exe


Saving to Google Drive (~5-10 min)...
Done!
-rw------- 1 root root 47M May 30 10:12 /content/drive/MyDrive/WRF/WRFV4.6.0/main/ndown.exe
-rw------- 1 root root 47M May 30 10:12 /content/drive/MyDrive/WRF/WRFV4.6.0/main/real.exe
-rw------- 1 root root 46M May 30 10:12 /content/drive/MyDrive/WRF/WRFV4.6.0/main/tc.exe
-rw------- 1 root root 55M May 30 10:12 /content/drive/MyDrive/WRF/WRFV4.6.0/main/wrf.exe


In [16]:
%%bash
cd /content
wget -q https://github.com/wrf-model/WPS/releases/download/v4.6.0/v4.6.0.tar.gz -O WPS.tar.gz
tar -xzf WPS.tar.gz
cd WPS-4.6.0

export WRF_DIR=/content/WRFV4.6.0
export NETCDF=/usr

echo '1' | ./configure
./compile >& wps_compile.log

if [ -f geogrid.exe ] && [ -f ungrib.exe ] && [ -f metgrid.exe ]; then
    echo 'WPS COMPILATION SUCCESS'
    cp -r /content/WPS-4.6.0 /content/drive/MyDrive/WRF/
    echo 'WPS saved to Drive'
else
    echo 'WPS FAILED'
    tail -20 wps_compile.log
fi

WPS FAILED
bash: line 10: ./compile: No such file or directory



gzip: stdin: unexpected end of file
tar: Child returned status 1
tar: Error is not recoverable: exiting now
bash: line 4: cd: WPS-4.6.0: No such file or directory
bash: line 9: ./configure: No such file or directory


In [17]:
%%bash
cd /content
rm -f WPS.tar.gz
rm -rf WPS-4.6.0

# Download with retry and verify
wget --tries=3 --timeout=60 \
  https://github.com/wrf-model/WPS/releases/download/v4.6.0/v4.6.0.tar.gz -O WPS.tar.gz

# Verify download
if gzip -t WPS.tar.gz 2>/dev/null; then
    echo "Download OK, extracting..."
    tar -xzf WPS.tar.gz
    ls WPS-4.6.0/
else
    echo "DOWNLOAD CORRUPT - re-run this cell"
fi

DOWNLOAD CORRUPT - re-run this cell


--2026-05-30 10:19:23--  https://github.com/wrf-model/WPS/releases/download/v4.6.0/v4.6.0.tar.gz
Resolving github.com (github.com)... 140.82.116.4
Connecting to github.com (github.com)|140.82.116.4|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-05-30 10:19:23 ERROR 404: Not Found.



In [18]:
%%bash
cd /content
rm -f WPS.tar.gz

wget --tries=3 https://github.com/wrf-model/WPS/releases/download/v4.5/v4.5.tar.gz -O WPS.tar.gz

if gzip -t WPS.tar.gz 2>/dev/null; then
    echo "Download OK"
    tar -xzf WPS.tar.gz
    ls
else
    echo "FAILED"
fi


FAILED


--2026-05-30 10:19:58--  https://github.com/wrf-model/WPS/releases/download/v4.5/v4.5.tar.gz
Resolving github.com (github.com)... 140.82.116.4
Connecting to github.com (github.com)|140.82.116.4|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-05-30 10:19:58 ERROR 404: Not Found.



In [19]:
%%bash
cd /content
rm -f WPS.tar.gz
rm -rf WPS-4.4

wget --tries=3 --timeout=120 \
  https://www2.mmm.ucar.edu/wrf/src/WPSV4.4.TAR.gz -O WPS.tar.gz

if gzip -t WPS.tar.gz 2>/dev/null; then
    echo "Download OK"
    tar -xzf WPS.tar.gz
    ls
else
    echo "FAILED - check output above"
fi

FAILED - check output above


--2026-05-30 10:20:29--  https://www2.mmm.ucar.edu/wrf/src/WPSV4.4.TAR.gz
Resolving www2.mmm.ucar.edu (www2.mmm.ucar.edu)... 128.117.13.91
Connecting to www2.mmm.ucar.edu (www2.mmm.ucar.edu)|128.117.13.91|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-05-30 10:20:29 ERROR 404: Not Found.



In [20]:
%%bash
cd /content
rm -f WPS.tar.gz
rm -rf WPS-4.4

wget --tries=3 --timeout=120 \
  https://github.com/wrf-model/WPS/archive/refs/tags/v4.4.tar.gz -O WPS.tar.gz

if gzip -t WPS.tar.gz 2>/dev/null; then
    echo "Download OK"
    tar -xzf WPS.tar.gz
    ls
else
    echo "FAILED"
fi

Download OK
drive
sample_data
v4.6.0.tar.gz
v4.6.0.tar.gz.1
WPS-4.4
wps_compile.log
WPS.tar.gz
WRFV4.6.0


--2026-05-30 10:21:05--  https://github.com/wrf-model/WPS/archive/refs/tags/v4.4.tar.gz
Resolving github.com (github.com)... 140.82.116.3
Connecting to github.com (github.com)|140.82.116.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://codeload.github.com/wrf-model/WPS/tar.gz/refs/tags/v4.4 [following]
--2026-05-30 10:21:06--  https://codeload.github.com/wrf-model/WPS/tar.gz/refs/tags/v4.4
Resolving codeload.github.com (codeload.github.com)... 140.82.116.9
Connecting to codeload.github.com (codeload.github.com)|140.82.116.9|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [application/x-gzip]
Saving to: ‘WPS.tar.gz’

     0K .......... .......... .......... .......... .......... 3.70M
    50K .......... .......... .......... .......... .......... 7.12M
   100K .......... .......... .......... .......... ..........  101M
   150K .......... .......... .......... .......... .......... 94.9M
   200K .......... ...

In [21]:
%%bash
cd /content/WPS-4.4

export WRF_DIR=/content/WRFV4.6.0
export NETCDF=/usr

echo '1' | ./configure

# Fix netcdf linking same as WRF
sed -i 's|-lnetcdf|-lnetcdf -lnetcdff|g' configure.wps

./compile >& wps_compile.log

if [ -f geogrid.exe ] && [ -f ungrib.exe ] && [ -f metgrid.exe ]; then
    echo 'WPS COMPILATION SUCCESS'
    cp -r /content/WPS-4.4 /content/drive/MyDrive/WRF/
    echo 'WPS saved to Drive'
else
    echo 'WPS FAILED'
    tail -20 wps_compile.log
fi

Will use NETCDF in dir: /usr
Using WRF I/O library in WRF build identified by $WRF_DIR: /content/WRFV4.6.0
$JASPERLIB or $JASPERINC not found in environment. Using default values for library paths...
------------------------------------------------------------------------
Please select from among the following supported platforms.

   1.  Linux x86_64, gfortran    (serial)
   2.  Linux x86_64, gfortran    (serial_NO_GRIB2)
   3.  Linux x86_64, gfortran    (dmpar)
   4.  Linux x86_64, gfortran    (dmpar_NO_GRIB2)
   5.  Linux x86_64, PGI compiler   (serial)
   6.  Linux x86_64, PGI compiler   (serial_NO_GRIB2)
   7.  Linux x86_64, PGI compiler   (dmpar)
   8.  Linux x86_64, PGI compiler   (dmpar_NO_GRIB2)
   9.  Linux x86_64, PGI compiler, SGI MPT   (serial)
  10.  Linux x86_64, PGI compiler, SGI MPT   (serial_NO_GRIB2)
  11.  Linux x86_64, PGI compiler, SGI MPT   (dmpar)
  12.  Linux x86_64, PGI compiler, SGI MPT   (dmpar_NO_GRIB2)
  13.  Linux x86_64, IA64 and Opteron    (serial)
  14

In [22]:
%%bash
ls -lh /content/WPS-4.4/geogrid.exe \
        /content/WPS-4.4/ungrib.exe \
        /content/WPS-4.4/metgrid.exe 2>/dev/null || echo "NOT FOUND"

# Also check as symlinks
ls -lh /content/WPS-4.4/*.exe 2>/dev/null


lrwxrwxrwx 1 root root 23 May 30 10:21 /content/WPS-4.4/geogrid.exe -> geogrid/src/geogrid.exe
lrwxrwxrwx 1 root root 23 May 30 10:22 /content/WPS-4.4/metgrid.exe -> metgrid/src/metgrid.exe
NOT FOUND
lrwxrwxrwx 1 root root 23 May 30 10:21 /content/WPS-4.4/geogrid.exe -> geogrid/src/geogrid.exe
lrwxrwxrwx 1 root root 23 May 30 10:22 /content/WPS-4.4/metgrid.exe -> metgrid/src/metgrid.exe


In [23]:
%%bash
# Check actual binaries behind symlinks
ls -lh /content/WPS-4.4/geogrid/src/geogrid.exe
ls -lh /content/WPS-4.4/metgrid/src/metgrid.exe

# Check ungrib error
grep -i "error\|failed" /content/WPS-4.4/wps_compile.log | tail -20


-rwxr-xr-x 1 root root 821K May 30 10:21 /content/WPS-4.4/geogrid/src/geogrid.exe
-rwxr-xr-x 1 root root 868K May 30 10:22 /content/WPS-4.4/metgrid/src/metgrid.exe
dec_jpeg2000.c:5:10: fatal error: jasper/jasper.h: No such file or directory
make[2]: [Makefile:75: dec_jpeg2000.o] Error 1 (ignored)
make[2]: [Makefile:57: libg2_4.a] Error 1 (ignored)
make[2]: [Makefile:58: libg2_4.a] Error 1 (ignored)
make[1]: [Makefile:10: g2/libg2_4.a] Error 1 (ignored)
collect2: error: ld returned 1 exit status
make[1]: [Makefile:19: ungrib.exe] Error 1 (ignored)
dec_jpeg2000.c:5:10: fatal error: jasper/jasper.h: No such file or directory
make[2]: [Makefile:75: dec_jpeg2000.o] Error 1 (ignored)
make[2]: [Makefile:57: libg2_4.a] Error 1 (ignored)
make[2]: [Makefile:58: libg2_4.a] Error 1 (ignored)
make[1]: [Makefile:10: g2/libg2_4.a] Error 1 (ignored)
dec_jpeg2000.c:5:10: fatal error: jasper/jasper.h: No such file or directory
make[2]: [Makefile:75: dec_jpeg2000.o] Error 1 (ignored)
make[2]: [Makefile:5

In [24]:
%%bash
apt-get install -y -q libjasper-dev libpng-dev

cd /content/WPS-4.4

export WRF_DIR=/content/WRFV4.6.0
export NETCDF=/usr
export JASPERLIB=/usr/lib/x86_64-linux-gnu
export JASPERINC=/usr/include

# Reconfigure with jasper
echo '1' | ./configure
sed -i 's|-lnetcdf|-lnetcdf -lnetcdff|g' configure.wps

./compile >& wps_compile.log

ls -lh geogrid.exe ungrib.exe metgrid.exe 2>/dev/null
ls -lh ungrib/src/ungrib.exe 2>/dev/null || echo "ungrib still missing"

Reading package lists...
Building dependency tree...
Reading state information...
Will use NETCDF in dir: /usr
Using WRF I/O library in WRF build identified by $WRF_DIR: /content/WRFV4.6.0
Found Jasper environment variables for GRIB2 support...
  $JASPERLIB = /usr/lib/x86_64-linux-gnu
  $JASPERINC = /usr/include
------------------------------------------------------------------------
Please select from among the following supported platforms.

   1.  Linux x86_64, gfortran    (serial)
   2.  Linux x86_64, gfortran    (serial_NO_GRIB2)
   3.  Linux x86_64, gfortran    (dmpar)
   4.  Linux x86_64, gfortran    (dmpar_NO_GRIB2)
   5.  Linux x86_64, PGI compiler   (serial)
   6.  Linux x86_64, PGI compiler   (serial_NO_GRIB2)
   7.  Linux x86_64, PGI compiler   (dmpar)
   8.  Linux x86_64, PGI compiler   (dmpar_NO_GRIB2)
   9.  Linux x86_64, PGI compiler, SGI MPT   (serial)
  10.  Linux x86_64, PGI compiler, SGI MPT   (serial_NO_GRIB2)
  11.  Linux x86_64, PGI compiler, SGI MPT   (dmpar)
  

E: Unable to locate package libjasper-dev


In [25]:
%%bash
apt-get install -y -q libjasper-files 2>/dev/null || \
apt-get install -y -q jasper 2>/dev/null || \
pip install -q jasper 2>/dev/null

# Alternative: use NO_GRIB2 version which skips jasper entirely
cd /content/WPS-4.4

export WRF_DIR=/content/WRFV4.6.0
export NETCDF=/usr

# Reconfigure with option 2 (serial NO_GRIB2) - skips jasper
echo '2' | ./configure
sed -i 's|-lnetcdf|-lnetcdf -lnetcdff|g' configure.wps
./compile >& wps_compile.log

ls -lh ungrib/src/ungrib.exe 2>/dev/null || echo "still missing"

Reading package lists...
Building dependency tree...
Reading state information...
Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  accountsservice app-install-data apparmor appmenu-gtk-module-common
  appmenu-gtk3-module appstream apt-config-icons apt-config-icons-hidpi
  apt-config-icons-large apt-config-icons-large-hidpi aspell aspell-en
  at-spi2-core avahi-daemon bluedevil bluez bluez-obexd bolt breeze
  breeze-cursor-theme breeze-gtk-theme breeze-icon-theme btrfs-progs
  bubblewrap bup bup-doc busybox-initramfs catdoc console-setup
  console-setup-linux cpio cryptsetup cryptsetup-bin cryptsetup-initramfs
  dbus-x11 dctrl-tools debconf-kde-data debconf-kde-helper
  device-tree-compiler dictionaries-common dmeventd dmidecode dmraid dmsetup
  docbook-xml docbook-xsl dosfstools dpkg-repack drkonqi efibootmgr eject
  fdisk fonts-hack fonts-noto-cjk fonts-noto-core fonts-noto-hinted
  fonts-noto-mono

In [26]:
%%bash
tail -30 /content/WPS-4.4/wps_compile.log | grep -A5 -i "ungrib"
ls -lh /content/WPS-4.4/ungrib/src/ungrib.exe 2>/dev/null || echo "ungrib exe missing"

ungrib exe missing


In [27]:
%%bash
apt-get install -y libjasper-dev 2>/dev/null || \
  (add-apt-repository -y ppa:rock-core/qt4 2>/dev/null && \
   apt-get install -y libjasper-dev) || \
  echo "jasper dev not available"

find /usr -name "jasper.h" 2>/dev/null

Reading package lists...
Building dependency tree...
Reading state information...
Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Get:6 http://security.ubuntu.com/ubuntu jammy-security/main amd64 DEP-11 Metadata [93.8 kB]
Get:7 http://security.ubuntu.com/ubuntu jammy-security/main DEP-11 48x48 Icons [21.6 kB]
Get:8 http://security.ubuntu.com/ubuntu jammy-security/main DEP-11 64x64 Icons [32.7 kB]
Get:9 http://security.ubuntu.com/ubuntu jammy-security/main DEP-11 64x64@2 Icons [29 B]
Hit:10 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Get:11 http://security.ubuntu.com/ubuntu jammy-security/main DEP-11 128x128 Icons [73.7 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 DEP-11 Metadata [157 B

E: Unable to locate package libjasper-dev


In [28]:
%%bash
cd /content

# Download and build jasper from source
wget -q https://github.com/jasper-software/jasper/releases/download/version-2.0.33/jasper-2.0.33.tar.gz
tar -xzf jasper-2.0.33.tar.gz
cd jasper-2.0.33
mkdir build && cd build
apt-get install -y -q cmake
cmake .. -DCMAKE_INSTALL_PREFIX=/usr/local \
         -DJAS_ENABLE_SHARED=ON \
         -DJAS_ENABLE_LIBJPEG=ON
make -j2
make install
ldconfig

# Verify
find /usr/local -name "jasper.h" 2>/dev/null
echo "Jasper done"%%bash
cd /content

# Download and build jasper from source
wget -q https://github.com/jasper-software/jasper/releases/download/version-2.0.33/jasper-2.0.33.tar.gz
tar -xzf jasper-2.0.33.tar.gz
cd jasper-2.0.33
mkdir build && cd build
apt-get install -y -q cmake
cmake .. -DCMAKE_INSTALL_PREFIX=/usr/local \
         -DJAS_ENABLE_SHARED=ON \
         -DJAS_ENABLE_LIBJPEG=ON
make -j2
make install
ldconfig

# Verify
find /usr/local -name "jasper.h" 2>/dev/null
echo "Jasper done"

Reading package lists...
Building dependency tree...
Reading state information...
cmake is already the newest version (3.22.1-1ubuntu1.22.04.2).
0 upgraded, 0 newly installed, 0 to remove and 8 not upgraded.
Jasper done%%bash
Reading package lists...
Building dependency tree...
Reading state information...
cmake is already the newest version (3.22.1-1ubuntu1.22.04.2).
0 upgraded, 0 newly installed, 0 to remove and 8 not upgraded.
Jasper done


mkdir: cannot create directory ‘build’: File exists
CMake Warning:
  Ignoring extra path from command line:

   ".."


CMake Error: The source directory "/content" does not appear to contain CMakeLists.txt.
Specify --help for usage, or press the help button on the CMake GUI.
make: *** No targets specified and no makefile found.  Stop.
make: *** No rule to make target 'install'.  Stop.
/sbin/ldconfig.real: /usr/local/lib/libtcm_debug.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind_2_5.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc.so.2 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtcm.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind_2_0.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libhwloc.so.15 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbb.so.12 is not a symbolic link



In [29]:
%%bash
cd /content/WPS-4.4

export WRF_DIR=/content/WRFV4.6.0
export NETCDF=/usr
export JASPERLIB=/usr/local/lib
export JASPERINC=/usr/local/include/jasper

echo '1' | ./configure
sed -i 's|-lnetcdf|-lnetcdf -lnetcdff|g' configure.wps

./compile >& wps_compile.log

ls -lh ungrib/src/ungrib.exe 2>/dev/null || echo "still missing"

Will use NETCDF in dir: /usr
Using WRF I/O library in WRF build identified by $WRF_DIR: /content/WRFV4.6.0
Found Jasper environment variables for GRIB2 support...
  $JASPERLIB = /usr/local/lib
  $JASPERINC = /usr/local/include/jasper
------------------------------------------------------------------------
Please select from among the following supported platforms.

   1.  Linux x86_64, gfortran    (serial)
   2.  Linux x86_64, gfortran    (serial_NO_GRIB2)
   3.  Linux x86_64, gfortran    (dmpar)
   4.  Linux x86_64, gfortran    (dmpar_NO_GRIB2)
   5.  Linux x86_64, PGI compiler   (serial)
   6.  Linux x86_64, PGI compiler   (serial_NO_GRIB2)
   7.  Linux x86_64, PGI compiler   (dmpar)
   8.  Linux x86_64, PGI compiler   (dmpar_NO_GRIB2)
   9.  Linux x86_64, PGI compiler, SGI MPT   (serial)
  10.  Linux x86_64, PGI compiler, SGI MPT   (serial_NO_GRIB2)
  11.  Linux x86_64, PGI compiler, SGI MPT   (dmpar)
  12.  Linux x86_64, PGI compiler, SGI MPT   (dmpar_NO_GRIB2)
  13.  Linux x86_64,

## Cell 4: Load WRF from Drive (Every Session)
Run this every session after first-time compilation.

In [ ]:
%%bash
echo 'Copying WRF from Drive to local (~2-3 min)...'
cp -r /content/drive/MyDrive/WRF/WRFV4.6.0 /content/
ls -lh /content/WRFV4.6.0/main/*.exe
echo 'WRF ready'

## Cell 5: Download WPS and Geography Data (First Time Only)

In [14]:
import os

if os.path.exists('/content/drive/MyDrive/WRF/WPS-4.6.0/geogrid.exe'):
    print('WPS already compiled on Drive. Skip this cell.')
else:
    print('Need to compile WPS')

Need to compile WPS


In [15]:
%%bash
# Download and compile WPS
cd /content
wget -q https://github.com/wrf-model/WPS/releases/download/v4.6.0/v4.6.0.tar.gz -O WPS.tar.gz
tar -xzf WPS.tar.gz
cd WPS-4.6.0

export WRF_DIR=/content/WRFV4.6.0
export NETCDF=/usr

echo '1' | ./configure
./compile >& wps_compile.log

if [ -f geogrid.exe ] && [ -f ungrib.exe ] && [ -f metgrid.exe ]; then
    echo 'WPS COMPILATION SUCCESS'
    cp -r /content/WPS-4.6.0 /content/drive/MyDrive/WRF/
    echo 'WPS saved to Drive'
else
    echo 'WPS FAILED'
    tail -20 wps_compile.log
fi

WPS FAILED
bash: line 11: ./compile: No such file or directory



gzip: stdin: unexpected end of file
tar: Child returned status 1
tar: Error is not recoverable: exiting now
bash: line 5: cd: WPS-4.6.0: No such file or directory
bash: line 10: ./configure: No such file or directory


In [ ]:
%%bash
# Download geography data (one time, ~2GB, saves to Drive)
if [ ! -d /content/drive/MyDrive/WRF/WPS_GEOG ]; then
    echo 'Downloading geography data (~2GB)...'
    wget -q https://www2.mmm.ucar.edu/wrf/src/wps_files/geog_high_res_mandatory.tar.gz -P /content/
    tar -xzf /content/geog_high_res_mandatory.tar.gz -C /content/
    cp -r /content/WPS_GEOG /content/drive/MyDrive/WRF/
    echo 'Geography data saved to Drive'
else
    echo 'Geography data already on Drive'
    cp -r /content/drive/MyDrive/WRF/WPS_GEOG /content/
fi

## Cell 6: Download Today's GFS Data
Run this every time you want a fresh forecast.

In [ ]:
%%bash
mkdir -p /content/GFS_data
DATE=$(date -u +%Y%m%d)
BASE="https://nomads.ncep.noaa.gov/pub/data/nccf/com/gfs/prod"

echo "Downloading GFS data for $DATE..."
for hr in 000 006 012 024; do
    URL="${BASE}/gfs.${DATE}/00/atmos/gfs.t00z.pgrb2.0p25.f${hr}"
    echo "Downloading f${hr}..."
    wget -q "$URL" -P /content/GFS_data/
done
echo 'GFS data downloaded'
ls -lh /content/GFS_data/

## Cell 7: Configure and Run WPS

In [ ]:
from datetime import datetime, timedelta

today = datetime.utcnow()
tomorrow = today + timedelta(days=1)

start = today.strftime('%Y-%m-%d_00:00:00')
end = tomorrow.strftime('%Y-%m-%d_00:00:00')

namelist_wps = f"""&share
 wrf_core = 'ARW',
 max_dom = 1,
 start_date = '{start}',
 end_date   = '{end}',
 interval_seconds = 21600,
/

&geogrid
 parent_id         = 1,
 parent_grid_ratio = 1,
 i_parent_start    = 1,
 j_parent_start    = 1,
 e_we  = 100,
 e_sn  = 100,
 geog_data_res = 'default',
 dx = 30000,
 dy = 30000,
 map_proj = 'lambert',
 ref_lat   = 19.0,
 ref_lon   = 72.8,
 truelat1  = 10.0,
 truelat2  = 30.0,
 stand_lon = 72.8,
 geog_data_path = '/content/WPS_GEOG',
/

&ungrib
 out_format = 'WPS',
 prefix = 'FILE',
/

&metgrid
 fg_name = 'FILE',
 io_form_metgrid = 2,
/
"""

with open('/content/WPS-4.6.0/namelist.wps', 'w') as f:
    f.write(namelist_wps)

print(f'namelist.wps written for {start} to {end}')

In [ ]:
%%bash
cd /content/WPS-4.6.0

# Run geogrid
./geogrid.exe >& geogrid.log
tail -1 geogrid.log

# Link GFS files and run ungrib
./link_grib.csh /content/GFS_data/gfs*
ln -sf ungrib/Variable_Tables/Vtable.GFS Vtable
./ungrib.exe >& ungrib.log
tail -1 ungrib.log

# Run metgrid
./metgrid.exe >& metgrid.log
tail -1 metgrid.log

echo 'WPS done'
ls met_em.d01.*.nc | head -5

## Cell 8: Configure and Run WRF

In [ ]:
from datetime import datetime, timedelta

today = datetime.utcnow()
tomorrow = today + timedelta(days=1)

namelist_input = f"""&time_control
 run_days                = 1,
 run_hours               = 0,
 run_minutes             = 0,
 run_seconds             = 0,
 start_year              = {today.year},
 start_month             = {today.month:02d},
 start_day               = {today.day:02d},
 start_hour              = 00,
 end_year                = {tomorrow.year},
 end_month               = {tomorrow.month:02d},
 end_day                 = {tomorrow.day:02d},
 end_hour                = 00,
 interval_seconds        = 21600,
 input_from_file         = .true.,
 history_interval        = 60,
 frames_per_outfile      = 1000,
 restart                 = .false.,
 restart_interval        = 7200,
 io_form_history         = 2,
 io_form_restart         = 2,
 io_form_input           = 2,
 io_form_boundary        = 2,
/

&domains
 time_step               = 180,
 max_dom                 = 1,
 e_we                    = 100,
 e_sn                    = 100,
 e_vert                  = 40,
 p_top_requested         = 5000,
 num_metgrid_levels      = 34,
 num_metgrid_soil_levels = 4,
 dx                      = 30000,
 dy                      = 30000,
 grid_id                 = 1,
 parent_id               = 0,
 i_parent_start          = 1,
 j_parent_start          = 1,
 parent_grid_ratio       = 1,
 parent_time_step_ratio  = 1,
 feedback                = 1,
 smooth_option           = 0,
/

&physics
 mp_physics              = 3,
 ra_lw_physics           = 1,
 ra_sw_physics           = 1,
 radt                    = 30,
 sf_sfclay_physics       = 1,
 sf_surface_physics      = 2,
 bl_pbl_physics          = 1,
 bldt                    = 0,
 cu_physics              = 1,
 cudt                    = 5,
 isfflx                  = 1,
 ifsnow                  = 1,
 icloud                  = 1,
 surface_input_source    = 3,
 num_soil_layers         = 4,
 sf_urban_physics        = 0,
/

&dynamics
 w_damping               = 0,
 diff_opt                = 1,
 km_opt                  = 4,
 diff_6th_opt            = 0,
 diff_6th_factor         = 0.12,
 base_temp               = 290.,
 damp_opt                = 0,
 zdamp                   = 5000.,
 dampcoef                = 0.2,
 khdif                   = 0,
 kvdif                   = 0,
 non_hydrostatic         = .true.,
 moist_adv_opt           = 1,
 scalar_adv_opt          = 1,
/

&bdy_control
 spec_bdy_width          = 5,
 spec_zone               = 1,
 relax_zone              = 4,
 specified              = .true.,
 nested                  = .false.,
/

&namelist_quilt
 nio_tasks_per_group = 0,
 nio_groups = 1,
/
"""

with open('/content/WRFV4.6.0/run/namelist.input', 'w') as f:
    f.write(namelist_input)

print('namelist.input written')

In [ ]:
%%bash
cd /content/WRFV4.6.0/run

# Link met_em files
ln -sf /content/WPS-4.6.0/met_em.d01.*.nc .

# Run real.exe
mpirun -np 2 ./real.exe >& real.log
tail -3 real.log

# Run wrf.exe (~20-40 min)
echo 'Starting wrf.exe...'
mpirun -np 4 ./wrf.exe >& wrf.log
echo 'WRF done'
ls -lh wrfout_d01_*

## Cell 9: Extract and Plot Temperature

In [ ]:
import xarray as xr
import glob
import numpy as np
import matplotlib.pyplot as plt

# Load all output files
files = sorted(glob.glob('/content/WRFV4.6.0/run/wrfout_d01_*'))
print(f'Found {len(files)} output files')

# Extract T2 (2m temperature) from each file
times = []
temps = []

for f in files:
    ds = xr.open_dataset(f)
    T2 = ds['T2'] - 273.15  # K to C
    # Center grid point
    cx, cy = T2.shape[2]//2, T2.shape[1]//2
    t = str(ds.Times.values[0].decode())
    temp = float(T2.isel(Time=0, south_north=cy, west_east=cx).values)
    times.append(t)
    temps.append(temp)
    print(f'{t} -> {temp:.1f}°C')
    ds.close()

# Plot
plt.figure(figsize=(12, 4))
plt.plot(range(len(temps)), temps, marker='o')
plt.xticks(range(len(times)), [t[11:16] for t in times], rotation=45)
plt.ylabel('Temperature (°C)')
plt.title('WRF 2m Temperature Forecast — Center Domain')
plt.grid(True)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/WRF/temperature_forecast.png')
plt.show()
print('Plot saved to Drive')